In [4]:
"""
استخراج النص من PDF (يدعم العربية)  |  Extract text from a PDF (Arabic-friendly)

  1) النص الموجود داخل الـ PDF (Text layer) عبر pypdf  -> سريع، والحروف دقيقة
  2) OCR (pytesseract) للصفحات التي بلا نص (ممسوحة ضوئياً)، أو بالقوة بـ --mode ocr

Install:
    pip install pypdf pytesseract pdf2image opencv-python numpy
    + Tesseract مع لغة العربية (للـ OCR فقط):
        Windows : https://github.com/UB-Mannheim/tesseract/wiki  (اختر Arabic أثناء التثبيت)
        Linux   : sudo apt install tesseract-ocr tesseract-ocr-ara poppler-utils
        Kaggle  : !apt-get install -y tesseract-ocr-ara poppler-utils
    (pdf2image يحتاج poppler: على Windows نزّله وأضف مجلد bin إلى PATH)

Examples:
    python pdf_text_extract.py                         # PDF_PATH، تلقائي (نص ثم OCR عند الحاجة)
    python pdf_text_extract.py --mode both             # ملفان: _text.txt و _ocr.txt لتقارن بينهما
    python pdf_text_extract.py --mode ocr --dpi 300    # OCR لكل الصفحات
    python pdf_text_extract.py --pages 1-3,10          # صفحات محددة
    python pdf_text_extract.py --tesseract "C:\\Program Files\\Tesseract-OCR\\tesseract.exe"
"""
import argparse
import os
import re
import shutil
import subprocess
import sys
import unicodedata

# ====== ضع مسار الـ PDF هنا (Put your PDF path here) ======
PDF_PATH = r"C:\Users\PC-LAB1\Downloads\على هامش السيرة_Foulabook.com_.pdf"
# ====== اسم ملف الخرج بدون امتداد ("" = اسم الـ PDF) ======
OUTPUT_NAME = ""
# =========================================================

_MARKS = re.compile("[\u202a-\u202e\u200e\u200f\u2066-\u2069\u200b\ufeff]")


def clean(text: str, keep_marks: bool = False) -> str:
    """Normalize Arabic presentation forms, drop invisible bidi marks, tidy whitespace."""
    text = unicodedata.normalize("NFKC", text or "")
    if not keep_marks:
        text = _MARKS.sub("", text)
    text = text.replace("\u00a0", " ")
    text = "\n".join(line.rstrip() for line in text.splitlines())
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def parse_pages(spec: str, total: int):
    if not spec:
        return list(range(1, total + 1))
    pages = set()
    for part in spec.split(","):
        part = part.strip()
        if "-" in part:
            a, b = part.split("-", 1)
            pages.update(range(int(a), int(b) + 1))
        elif part:
            pages.add(int(part))
    return sorted(p for p in pages if 1 <= p <= total)


# ----------------------------------------------------------------- text layer
def extract_text_layer(reader, page_no: int) -> str:
    return reader.pages[page_no - 1].extract_text() or ""


# ------------------------------------------------------------------------ OCR
class OCR:
    def __init__(self, pdf, lang="ara+eng", dpi=300, psm=6, preprocess=True, tesseract=None):
        import pytesseract
        self.pt, self.pdf, self.lang, self.dpi, self.psm, self.pre = pytesseract, pdf, lang, dpi, psm, preprocess
        cmd = tesseract or shutil.which("tesseract")
        if not cmd or not os.path.exists(cmd):
            raise RuntimeError("Tesseract غير مثبت أو غير موجود في PATH. استخدم --tesseract بمسار tesseract.exe")
        pytesseract.pytesseract.tesseract_cmd = cmd
        have = set(pytesseract.get_languages(config=""))
        want = [l for l in lang.split("+") if l]
        missing = [l for l in want if l not in have]
        if missing:
            raise RuntimeError(f"لغة OCR غير مثبتة: {missing}. المتوفر: {sorted(have)}. ثبّت حزمة 'ara' (انظر أعلى الملف).")

    def page(self, page_no: int) -> str:
        from pdf2image import convert_from_path
        img = convert_from_path(self.pdf, dpi=self.dpi, first_page=page_no, last_page=page_no)[0]
        if self.pre:
            import cv2
            import numpy as np
            g = cv2.GaussianBlur(np.array(img.convert("L")), (3, 3), 0)
            _, g = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            img = g
        return self.pt.image_to_string(img, lang=self.lang, config=f"--oem 1 --psm {self.psm}")


# ----------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(allow_abbrev=False)
    ap.add_argument("--pdf", default=PDF_PATH)
    ap.add_argument("--mode", choices=["auto", "text", "ocr", "both"], default="auto",
                    help="auto: text layer, OCR only for empty pages | text | ocr | both: write two files")
    ap.add_argument("--pages", default="", help="e.g. 1-3,7 (default: all)")
    ap.add_argument("--lang", default="ara+eng", help="Tesseract languages")
    ap.add_argument("--dpi", type=int, default=300)
    ap.add_argument("--psm", type=int, default=6, help="Tesseract page segmentation (6 block, 4 columns, 3 auto)")
    ap.add_argument("--no-preprocess", action="store_true", help="skip grayscale/threshold before OCR")
    ap.add_argument("--min-chars", type=int, default=20, help="auto mode: below this a page counts as empty")
    ap.add_argument("--tesseract", default=None, help="path to tesseract executable")
    ap.add_argument("--keep-marks", action="store_true", help="keep invisible bidi marks")
    ap.add_argument("--out", default=None, help="output .txt (default: <pdf name>.txt)")
    ap.add_argument("--no-print", action="store_true", help="do not print the text on screen")
    ap.add_argument("--max-print", type=int, default=0, help="limit characters printed per page (0 = all)")
    ap.add_argument("--open", action="store_true", help="open the saved file automatically")
    args, _ = ap.parse_known_args()  # works in Jupyter / Kaggle too

    if not os.path.isfile(args.pdf):
        sys.exit(f"File not found: {args.pdf}")
    from pypdf import PdfReader
    reader = PdfReader(args.pdf)
    pages = parse_pages(args.pages, len(reader.pages))
    base = OUTPUT_NAME or os.path.splitext(os.path.basename(args.pdf))[0]
    print(f"{args.pdf}: {len(reader.pages)} pages, processing {len(pages)}  (mode={args.mode})")

    ocr, ocr_error = None, None
    if args.mode in ("auto", "ocr", "both"):
        try:
            ocr = OCR(args.pdf, args.lang, args.dpi, args.psm, not args.no_preprocess, args.tesseract)
        except Exception as e:  # keep going with the text layer if OCR is unavailable
            ocr_error = str(e)
            if args.mode in ("ocr", "both"):
                sys.exit(f"OCR error: {ocr_error}")
            print(f"WARNING (OCR unavailable): {ocr_error}")

    results = {"text": [], "ocr": [], "final": []}
    for n in pages:
        t = clean(extract_text_layer(reader, n), args.keep_marks) if args.mode != "ocr" else ""
        o = clean(ocr.page(n), args.keep_marks) if (ocr and args.mode in ("ocr", "both")) else ""
        if args.mode == "auto":
            if len(t) >= args.min_chars:
                src, f = "text layer", t
            elif ocr:
                f = clean(ocr.page(n), args.keep_marks)
                src = "OCR"
            else:
                src, f = "EMPTY (no text, OCR unavailable)", t
        elif args.mode == "text":
            src, f = "text layer", t
        elif args.mode == "ocr":
            src, f = "OCR", o
        else:
            src, f = "both", t
        print(f"  page {n:3d}: {src:34s} {len(f) if args.mode != 'both' else str((len(t), len(o)))} chars")
        results["text"].append((n, t))
        results["ocr"].append((n, o))
        results["final"].append((n, f))

    def write(path, items):
        with open(path, "w", encoding="utf-8-sig") as fh:  # BOM so Arabic shows correctly in Notepad/Excel
            for n, txt in items:
                fh.write(f"===== صفحة {n} / Page {n} =====\n{txt}\n\n")
        return os.path.abspath(path)

    saved = []
    if args.mode == "both":
        saved = [write(f"{base}_text.txt", results["text"]), write(f"{base}_ocr.txt", results["ocr"])]
        shown = [("TEXT LAYER", results["text"]), ("OCR", results["ocr"])]
    else:
        saved = [write(args.out or f"{base}.txt", results["final"])]
        shown = [(None, results["final"])]

    if not args.no_print:  # show the result right here, no need to hunt for the file
        for title, items in shown:
            print("\n" + "=" * 60 + (f"\n  {title}\n" + "=" * 60 if title else ""))
            for n, txt in items:
                body = txt[:args.max_print] + " ..." if args.max_print and len(txt) > args.max_print else txt
                print(f"\n----- صفحة {n} / Page {n} -----\n{body}")
    print("\nSaved files:\n  " + "\n  ".join(saved))
    if args.open and saved:
        try:
            os.startfile(saved[0])  # Windows
        except AttributeError:
            subprocess.run(["open" if sys.platform == "darwin" else "xdg-open", saved[0]])
    return "\n\n".join(t for _, t in results["final"])


if __name__ == "__main__":
    main()

C:\Users\PC-LAB1\Downloads\على هامش السيرة_Foulabook.com_.pdf: 420 pages, processing 420  (mode=auto)
WARNING (OCR unavailable): Tesseract غير مثبت أو غير موجود في PATH. استخدم --tesseract بمسار tesseract.exe
  page   1: EMPTY (no text, OCR unavailable)   0 chars
  page   2: EMPTY (no text, OCR unavailable)   13 chars
  page   3: EMPTY (no text, OCR unavailable)   0 chars
  page   4: text layer                         26 chars
  page   5: text layer                         803 chars
  page   6: text layer                         242 chars
  page   7: text layer                         272 chars
  page   8: text layer                         1169 chars
  page   9: text layer                         1555 chars
  page  10: text layer                         1781 chars
  page  11: text layer                         1759 chars
  page  12: text layer                         742 chars
  page  13: EMPTY (no text, OCR unavailable)   0 chars
  page  14: EMPTY (no text, OCR unavailable)   12 char

In [5]:
REMOVE_DIRECTION_MARKS = True
MANUAL_FIXES = {                      # "wrong": "right"  -> extend after proofreading the output
    "حفر رمزم": "حفر زمزم",
    "يطمثن": "يطمئن",
    "الشرقء": "الشرق،",
    "أعسرء": "أعسر،",
    "والظواهرء": "والظواهر،",
}
AR = r"[\u0621-\u064A]"

def clean_text(raw):
    t = raw
    if REMOVE_DIRECTION_MARKS:
        t = re.sub(r"[\u200e\u200f\u202a-\u202e\u2066-\u2069]", "", t)

    # Arabic comma misread as . ؛ , » inside / at the end of a sentence
    t = re.sub(rf"[.؛,]+[ \t]*(?:،[ \t]*)?(?=[ \t]+{AR})", "،", t)       # only when more words follow on the line
    t = re.sub(rf"(?<={AR})[؛,»]+[ \t]*$", "،", t, flags=re.M)             # at line end
    t = re.sub(rf"(?<={AR})»(?=[ \t])", "،", t)                            # "العلم»" -> "العلم،"
    t = re.sub(r"[؛.,]+ء", "،", t)
    t = re.sub(r"[.؛,]*؛[ \t]*$", "،", t, flags=re.M)
    t = re.sub(r"(?<=[\u064B-\u0652])ء(?=[ \t]|$)", "،", t, flags=re.M)    # ء right after a vowel mark
    t = re.sub(r"(?<=\u064B\u0627)ء(?=[ \t]|$)", "،", t, flags=re.M)        # ء after "ًا"
    t = re.sub(r"،{2,}", "،", t)
    t = re.sub(r"[ \t]*،", "،", t)
    t = re.sub(r"،(?=[^\s،])", "، ", t)

    # a trailing ء that is really a comma: the word without it is common here, the word with it is not
    words = Counter(re.findall(rf"{AR}+", t))
    def fix_hamza(m):
        w = m.group(0); base = w[:-1]
        return base + "،" if words[base] >= 2 and words[w] <= 1 else w
    t = re.sub(rf"{AR}+ء(?=[ \t]|$)", fix_hamza, t, flags=re.M)

    for wrong, right in MANUAL_FIXES.items():
        t = t.replace(wrong, right)
    t = re.sub(r"[ \t]{2,}", " ", t)
    t = re.sub(r"[ \t]+$", "", t, flags=re.M)
    return re.sub(r"\n{3,}", "\n\n", t).strip()

clean = clean_text(raw_text)
OUTPUT_TXT.write_text(clean, encoding="utf-8")
print(f"Cleaned text saved to {OUTPUT_TXT.resolve()}  ({len(clean):,} characters)\n")
print(clean[:1500])

NameError: name 'raw_text' is not defined